# Ingest Dimension Data into the Bronze Layer

This notebook reads raw CSV data for all dimension tables from the source volume, applies explicit schemas, adds metadata columns for lineage tracking, and writes the ingested data as managed Delta tables in the Bronze layer of the e-commerce medallion architecture.

## Dimensions Ingested

| Dimension | Source CSV Path | Bronze Table | Row Count |
| --- | --- | --- | --- |
| Brands | `raw/brands/*.csv` | `ecommerce.bronze.brz_brands` | 52 |
| Category | `raw/category/*.csv` | `ecommerce.bronze.brz_category` | 10 |
| Customers | `raw/customers/*.csv` | `ecommerce.bronze.brz_customers` | 300,000 |
| Date | `raw/date/*.csv` | `ecommerce.bronze.brz_date` | 95 |
| Products | `raw/products/*.csv` | `ecommerce.bronze.brz_products` | 50,000 |

All source files reside under `/Volumes/ecommerce/source_data/raw/`.

## Standard Processing Pattern

Each dimension follows the same four-step pipeline:

1. **Define schema** — explicit `StructType` with `StringType` fields ensures type safety and rejects malformed data
2. **Read CSV** — `spark.read` with `header=True`, `delimiter=","`, and the defined schema (no schema inference)
3. **Add metadata** — `_source_file` (from Spark's built-in `_metadata.file_path`) and `_ingested_at` (current timestamp) for lineage tracking
4. **Write to Bronze** — `saveAsTable` as a managed Delta table with `mode("overwrite")` and `mergeSchema=True`

In [0]:
# Import required libraries for schema definition and data ingestion
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)
import pyspark.sql.functions as F

In [0]:
# Define the catalog name for the e-commerce project
catalog_name = "ecommerce"

In [0]:
# Set the Spark session time zone to UTC for consistent timestamp handling

spark.conf.set("spark.sql.session.timeZone", "UTC")

### Brands Dim Processing

Ingests raw brand data with columns: `brand_code`, `brand_name`, `category_code`. The resulting Bronze table `ecommerce.bronze.brz_brands` will have 5 columns (3 data + 2 metadata) and 52 rows.

In [0]:
# Define the schema for the brands dimension table
brand_schema = StructType(
    [
        StructField("brand_code", StringType(), False),
        StructField("brand_name", StringType(), False),
        StructField("category_code", StringType(), False),
    ]
)

# Raw brands data file path
brand_raw_path = "/Volumes/ecommerce/source_data/raw/brands/*.csv"

# Read the raw brands CSV data into a DataFrame with the defined schema
df_brand = (
    spark.read.option("delimiter", ",")
    .option("header", True)
    .schema(brand_schema)
    .csv(brand_raw_path)
)

df_brand.show(5)

In [0]:
# Display the schema of the brands DataFrame
df_brand.printSchema()

In [0]:
# Add metadata columns for lineage tracking:
#   _source_file  → extracted from Spark's built-in _metadata.file_path (the CSV file each row came from)
#   _ingested_at  → current timestamp marking when the data was ingested into the Bronze layer
df_brand = df_brand.withColumn("_source_file", F.col("_metadata.file_path")).withColumn(
    "_ingested_at", F.current_timestamp()
)

df_brand.show(5, truncate=False)

In [0]:
# Display the row count, column count, and schema of the brands DataFrame
print(f"\nNumber of rows: {df_brand.count()}")
print(f"Number of columns: {len(df_brand.columns)}\n")

df_brand.printSchema()

**Write the Brands dimension table to the Bronze layer**

The raw brands CSV data has been read with an explicit schema and metadata columns have been added for lineage tracking. We can now write the DataFrame to the Bronze layer as a managed Delta table (`brz_brands`).

In [0]:
# Write the brands DataFrame as a Delta table in the bronze layer
df_brand.write.mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", True) \
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

### Category Dim Processing

Ingests raw category data with columns: `category_code`, `category_name`. The resulting Bronze table `ecommerce.bronze.brz_category` will have 4 columns (2 data + 2 metadata) and 10 rows.

In [0]:
# Define the schema for the category dimension table
category_schema = StructType(
    [
        StructField("category_code", StringType(), False),
        StructField("category_name", StringType(), False),
    ]
)

# Raw category data file path
category_raw_path = "/Volumes/ecommerce/source_data/raw/category/*.csv"

In [0]:
# Read the raw category CSV data into a DataFrame with the defined schema
df_category = (
    spark.read.option("delimiter", ",")
    .option("header", True)
    .schema(category_schema)
    .csv(category_raw_path)
)

df_category.show(5)

In [0]:
# Add metadata columns for lineage tracking
df_category = df_category.withColumn(
    "_source_file", F.col("_metadata.file_path")
).withColumn("_ingested_at", F.current_timestamp())

df_category.show(5, truncate=False)

In [0]:
# Display the row count, column count, and schema of the category DataFrame
print(f"Number of rows: {df_category.count()}")
print(f"Number of columns: {len(df_category.columns)}\n")

df_category.printSchema()

**Write the Category dimension table to the Bronze layer**

The raw category CSV data has been read with an explicit schema and metadata columns have been added for lineage tracking. We can now write the DataFrame to the Bronze layer as a managed Delta table (`brz_category`).

In [0]:
# Write the category DataFrame as a Delta table in the bronze layer

df_category.write.mode("overwrite").format("delta").option(
    "mergeSchema", True
).saveAsTable(f"{catalog_name}.bronze.brz_category")

### Customers Dim Processing

Ingests raw customer data with columns: `customer_id`, `phone`, `country_code`, `country`, `state`. The resulting Bronze table `ecommerce.bronze.brz_customers` will have 7 columns (5 data + 2 metadata) and 300,000 rows.

In [0]:
# Define the schema for the customers dimension table
customers_schema = StructType(
    [
        StructField("customer_id", StringType(), False),
        StructField("phone", StringType(), True),
        StructField("country_code", StringType(), True),
        StructField("country", StringType(), True),
        StructField("state", StringType(), True),
    ]
)

# Raw customers data file path
customer_raw_path = "/Volumes/ecommerce/source_data/raw/customers/*.csv"

In [0]:
# Read the raw customers CSV data into a DataFrame with the defined schema
df_customers = (
    spark.read.option("header", True)
    .option("delimiter", ",")
    .schema(customers_schema)
    .csv(customer_raw_path)
)

df_customers.show(5, truncate=False)

In [0]:
# Add metadata columns for lineage tracking
df_customers = df_customers.withColumn(
    "_source_file", F.col("_metadata.file_path")
).withColumn("_ingested_at", F.current_timestamp())

df_customers.show(5, truncate=False)

In [0]:
# Display the row count, column count, and schema of the customers DataFrame
print(f"Number of rows: {df_customers.count()}")
print(f"Number of columns: {len(df_customers.columns)}\n")

df_customers.printSchema()

**Write the Customers dimension table to the Bronze layer**

The raw customers CSV data has been read with an explicit schema and metadata columns have been added for lineage tracking. We can now write the DataFrame to the Bronze layer as a managed Delta table (`brz_customers`).

In [0]:
# Write the customers DataFrame as a Delta table in the bronze layer

df_customers.write.mode("overwrite").format("delta").option(
    "mergeSchema", True
).saveAsTable(f"{catalog_name}.bronze.brz_customers")

### Date Dim Processing

Ingests raw date data with columns: `date`, `year`, `day_name`, `quarter`, `week_of_year`. The resulting Bronze table `ecommerce.bronze.brz_date` will have 7 columns (5 data + 2 metadata) and 95 rows.

In [0]:
# Define the schema for the date dimension table
date_schema = StructType(
    [
        StructField("date", StringType(), False),
        StructField("year", StringType(), True),
        StructField("day_name", StringType(), True),
        StructField("quarter", StringType(), True),
        StructField("week_of_year", StringType(), True),
    ]
)

# Raw date data file path
date_raw_path = "/Volumes/ecommerce/source_data/raw/date/*.csv"

In [0]:
# Read the raw date CSV data into a DataFrame and add metadata columns
df_date = (
    spark.read.option("header", True)
    .option("delimiter", ",")
    .schema(date_schema)
    .csv(date_raw_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_date.show(5, truncate=False)

In [0]:
# Display the row count, column count, and schema of the date DataFrame
print(f"Number of rows: {df_date.count()}")
print(f"Number of columns: {len(df_date.columns)}\n")

df_date.printSchema()

**Write the Date dimension table to the Bronze layer**

The raw date CSV data has been read with an explicit schema and metadata columns have been added for lineage tracking. We can now write the DataFrame to the Bronze layer as a managed Delta table (`brz_date`).

In [0]:
# Write the date DataFrame as a Delta table in the bronze layer

df_date.write.mode("overwrite").format("delta").option("mergeSchema", True).saveAsTable(
    f"{catalog_name}.bronze.brz_date"
)

### Products Dim Processing

Ingests raw product data with columns: `product_id`, `sku`, `category_code`, `brand_code`, `color`, `size`, `material`, `weight_grams`, `length_cms`, `width_cm`, `height_cms`, `rating_count`. The resulting Bronze table `ecommerce.bronze.brz_products` will have 14 columns (12 data + 2 metadata) and 50,000 rows.

In [0]:
# Define the schema for the products dimension table

products_schema = StructType(
    [
        StructField("product_id", StringType(), False),
        StructField("sku", StringType(), True),
        StructField("category_code", StringType(), True),
        StructField("brand_code", StringType(), True),
        StructField("color", StringType(), True),
        StructField("size", StringType(), True),
        StructField("material", StringType(), True),
        StructField("weight_grams", StringType(), True),
        StructField("length_cms", StringType(), True),
        StructField("width_cm", StringType(), True),
        StructField("height_cms", StringType(), True),
        StructField("rating_count", StringType(), True),
    ]
)

# Raw products data file path
products_raw_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

In [0]:
# Read the raw products CSV data into a DataFrame and add metadata columns
df_products = (
    spark.read.option("header", True)
    .option("delimiter", ",")
    .schema(products_schema)
    .csv(products_raw_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_products.show(5, truncate=False)

In [0]:
# Display the row count, column count, and schema of the products DataFrame
print(f"Number of rows: {df_products.count()}")
print(f"Number of columns: {len(df_products.columns)}\n")

df_products.printSchema()

**Write the Products dimension table to the Bronze layer**

The raw products CSV data has been read with an explicit schema and metadata columns have been added for lineage tracking. We can now write the DataFrame to the Bronze layer as a managed Delta table (`brz_products`).

In [0]:
# Write the products DataFrame as a Delta table in the bronze layer
df_products.write.mode("overwrite").option("mergeSchema", True).format(
    "delta"
).saveAsTable(f"{catalog_name}.bronze.brz_products")

## Summary

All five dimension tables have been ingested into the Bronze layer:

| Bronze Table | Rows | Data Columns | Metadata Columns |
| --- | --- | --- | --- |
| `ecommerce.bronze.brz_brands` | 52 | 3 | 2 |
| `ecommerce.bronze.brz_category` | 10 | 2 | 2 |
| `ecommerce.bronze.brz_customers` | 300,000 | 5 | 2 |
| `ecommerce.bronze.brz_date` | 95 | 5 | 2 |
| `ecommerce.bronze.brz_products` | 50,000 | 12 | 2 |

Each table includes `_source_file` and `_ingested_at` metadata columns for lineage tracking. All data columns are stored as `StringType` at this stage — type casting and cleansing will happen in the Silver layer.